# Phase 3 Session 2 — NB4: Evaluate

**Input:**
- `semeval-absa-restaurant`: SemEval XML files
- `p3s2-embedding`: embedding checkpoint + processed data (from nb1)
- `p3s2-index`: FAISS index (from nb2)
- `p3s2-absa`: ABSA checkpoint (from nb3)

**Baseline:** Phase 2 no-retrieval Joint F1 = 0.6104

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import glob
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# SemEval data
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/semeval-2016-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
if KAGGLE_INPUT is None:
    inputs = glob.glob('/kaggle/input/**/semeval*train*.xml', recursive=True)
    assert inputs, 'Cannot find SemEval dataset'
    KAGGLE_INPUT = os.path.dirname(inputs[0])

print(f'Using: {KAGGLE_INPUT}')
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)
train_xml = glob.glob(f'{KAGGLE_INPUT}/*[Tt]rain*.xml')
test_xml = glob.glob(f'{KAGGLE_INPUT}/*[Tt]est*.xml') + glob.glob(f'{KAGGLE_INPUT}/*gold*')
shutil.copy(train_xml[0], 'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(test_xml[0], 'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')
print('XML files in place.')

In [ ]:
NB1_INPUT = None
for candidate in ['/kaggle/input/p3s2-embedding',
                  '/kaggle/input/datasets/lcminhc/p3s2-embedding']:
    if os.path.exists(candidate):
        NB1_INPUT = candidate
        break
NB2_INPUT = None
for candidate in ['/kaggle/input/p3s2-index',
                  '/kaggle/input/datasets/lcminhc/p3s2-index']:
    if os.path.exists(candidate):
        NB2_INPUT = candidate
        break
NB3_INPUT = None
for candidate in ['/kaggle/input/p3s2-absa',
                  '/kaggle/input/datasets/lcminhc/p3s2-absa']:
    if os.path.exists(candidate):
        NB3_INPUT = candidate
        break
assert NB1_INPUT, 'Dataset p3s2-embedding not found'
assert NB2_INPUT, 'Dataset p3s2-index not found'
assert NB3_INPUT, 'Dataset p3s2-absa not found'
print(f'NB1: {NB1_INPUT}')
print(f'NB2: {NB2_INPUT}')
print(f'NB3: {NB3_INPUT}')

# Copy embedding checkpoint
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{NB1_INPUT}/embedding_best.pt', 'checkpoints/embedding/best.pt')

# Copy processed data (flat layout)
os.makedirs('data/processed', exist_ok=True)
for f in ['bio_tagging.jsonl', 'classification.jsonl', 'contrastive_triplets.jsonl']:
    shutil.copy(f'{NB1_INPUT}/{f}', f'data/processed/{f}')

# Copy index (flat layout)
os.makedirs('indexes', exist_ok=True)
for f in os.listdir(NB2_INPUT):
    if f.endswith(('.faiss', '.jsonl', '.npy')):
        shutil.copy(f'{NB2_INPUT}/{f}', f'indexes/{f}')

# Copy ABSA checkpoint
os.makedirs('checkpoints/absa', exist_ok=True)
shutil.copy(f'{NB3_INPUT}/absa_best.pt', 'checkpoints/absa/best.pt')

print('All artifacts loaded:')
print(f'  Embedding: {os.path.getsize("checkpoints/embedding/best.pt") / 1e6:.1f} MB')
print(f'  ABSA:      {os.path.getsize("checkpoints/absa/best.pt") / 1e6:.1f} MB')
print(f'  Index:     {os.path.getsize("indexes/train.faiss") / 1e6:.1f} MB')

## 1. Evaluate (with Retrieval)

In [ ]:
!python scripts/05_evaluate.py \
    --config configs/absa.yaml \
    --checkpoint checkpoints/absa/best.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --retrieval_config configs/retrieval_v2.yaml

## 2. Compare with Baselines

In [ ]:
print('=== Results Comparison ===')
print(f'{"Metric":<20} {"P2 retrieval":<14} {"P2 no-ret":<14} {"S2 retrieval":<14}')
print('-' * 62)
baselines = {
    'BIO Token F1':    (0.5418, 0.6198),
    'Span F1':         (0.6074, 0.6489),
    'Sentiment Acc':   (0.8976, 0.9243),
    'Sentiment MacF1': (0.7589, 0.8234),
    'Joint F1':        (0.5460, 0.6104),
}
for metric, (p2_ret, p2_noret) in baselines.items():
    print(f'{metric:<20} {p2_ret:<14.4f} {p2_noret:<14.4f} {"see above":<14}')

print()
print('Goal: S2 retrieval Joint F1 > 0.6104 (Phase 2 no-ret baseline)')

## 3. Save Results

In [ ]:
output_dir = '/kaggle/working/outputs_p3s2_nb4'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs/eval_results.md'):
    shutil.copy('logs/eval_results.md', os.path.join(output_dir, 'eval_results.md'))
    with open('logs/eval_results.md') as f:
        print(f.read())

print(f'Results saved to {output_dir}')